In [1]:
pip install --upgrade google-cloud-storage


  Attempting uninstall: google-resumable-media
    Found existing installation: google-resumable-media 1.3.1
    Uninstalling google-resumable-media-1.3.1:
      Successfully uninstalled google-resumable-media-1.3.1
  Attempting uninstall: google-cloud-core
    Found existing installation: google-cloud-core 1.7.1
    Uninstalling google-cloud-core-1.7.1:
      Successfully uninstalled google-cloud-core-1.7.1
  Attempting uninstall: google-cloud-storage
    Found existing installation: google-cloud-storage 1.31.0
    Uninstalling google-cloud-storage-1.31.0:
      Successfully uninstalled google-cloud-storage-1.31.0


In [4]:
pip install --upgrade google-cloud-speech

In [2]:
# Import os
import os
# Imports the Google Cloud client library
from google.cloud import storage
from google.cloud import speech
import pprint

#constants

BUCKET_NAME = "admn5015_cs"

# Create environment variable for Google Cloud credentials
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "credentials"+ os.sep+"admn5015-w23-in-class-376513-04be5a8f747c.json"

def list_blobs(bucket_name):
    """Lists all the blobs in the bucket."""
    # bucket_name = "your-bucket-name"

    storage_client = storage.Client()

    # Note: Client.list_blobs requires at least package version 1.17.0.
    blobs = storage_client.list_blobs(bucket_name)

    return blobs

    # Note: The call returns a response only when the iterator is consumed.
    # for blob in blobs:
    # print(blob.name)

def extract_audio_gcs(gcs_uri):
    """Asynchronously transcribes the audio file specified by the gcs_uri."""

    client = speech.SpeechClient()

    audio = speech.RecognitionAudio(uri=gcs_uri)
    config = speech.RecognitionConfig(
        encoding=speech.RecognitionConfig.AudioEncoding.LINEAR16,
        audio_channel_count=2,
        language_code="en-US",
    )

    operation = client.long_running_recognize(config=config, audio=audio)

    print("Waiting for operation to complete...")
    response = operation.result(timeout=600)

    return response

def main():

    blobs = list_blobs(BUCKET_NAME)

    # Extract audio from each file
    for blob in blobs:

        gcs_uri = "gs://" + BUCKET_NAME + "/" + blob.name
        print("\nThis is the uri: {}".format(gcs_uri))
        

        response = extract_audio_gcs(gcs_uri=gcs_uri)

        for result in response.results:
            print("\nTranscript: {}".format(result.alternatives[0].transcript))
            print("Confidence: {}".format(result.alternatives[0].confidence))
    
    
    
    if __name__ =="__main__":
        main()
